# 03 - ONNX export + TensorRT benchmark on a Colab T4

In [ ]:
!git clone https://github.com/<YOUR_USER>/adpstack.git
%cd adpstack
!pip install -q -e ".[ml,export]" tensorrt onnxruntime-gpu

In [ ]:
!python -m adpstack.export.to_onnx --yolo /content/drive/MyDrive/adp_models/yolo11n_best.pt --midas --out outputs/onnx

In [ ]:
ONNX='/content/drive/MyDrive/adp_models/yolo11n_best.onnx'
!python -m adpstack.export.to_tensorrt --onnx $ONNX --precision fp16 --out outputs/yolo_fp16.engine
!python -m adpstack.export.to_tensorrt --onnx $ONNX --precision int8 --calib-dir /content/bdd_yolo/images/val --out outputs/yolo_int8.engine

In [ ]:
!python -m adpstack.export.benchmark --torch-weights /content/drive/MyDrive/adp_models/yolo11n_best.pt --onnx $ONNX --engines trt_fp16=outputs/yolo_fp16.engine trt_int8=outputs/yolo_int8.engine --iters 500 --wandb

## INT8 accuracy drop
Run `yolo val model=outputs/yolo_int8.engine data=... ` (Ultralytics supports .engine) and record mAP50-95 vs FP32 in docs/results.md.